# QR Code Generator - Clean / No Noise Version

This notebook generates QR codes for the English-German Vocabulary Book.

- No noise is added.
- QR codes are black and white PNG images.
- Each QR has a different `uid` URL parameter.
- A CSV mapping file is also created.
- Optional OpenCV decode check is included.


In [ ]:
!pip install -q "qrcode[pil]"

In [ ]:
import os
import zipfile
import secrets
from pathlib import Path
from urllib.parse import urlencode

import pandas as pd
import qrcode
from PIL import Image
import matplotlib.pyplot as plt

try:
    import cv2
    CV2_AVAILABLE = True
except Exception:
    CV2_AVAILABLE = False

from google.colab import files

In [ ]:
# Change this URL to your own GitHub Pages URL.
BASE_URL = "https://bokuhabobu.github.io/present/"

# Manual user IDs.
USER_IDS = [
    "u_001",
    "u_002",
    "u_003",
    "u_004",
    "u_005",
]

# Optional: random user IDs.
# USER_IDS = [f"u_{secrets.token_hex(3)}" for _ in range(20)]

OUTPUT_DIR = Path("qr_codes_clean")
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
def sanitize_uid(uid):
    """Keep only safe characters for filename / URL parameter."""
    safe = "".join(ch for ch in str(uid).strip() if ch.isalnum() or ch in ["_", "-"])
    return safe[:40] or "default"


def make_user_url(base_url, uid):
    """Create GitHub Pages URL with uid parameter."""
    base_url = base_url.strip()
    separator = "&" if "?" in base_url else "?"
    return f"{base_url}{separator}{urlencode({'uid': uid})}"


def create_clean_qr(url, save_path):
    """
    Create a readable QR code without noise.

    Clean QR settings:
    - black / white only
    - large box_size
    - enough border
    - RGB PNG output
    - high error correction
    """
    qr = qrcode.QRCode(
        version=None,
        error_correction=qrcode.constants.ERROR_CORRECT_H,
        box_size=12,
        border=5,
    )

    qr.add_data(url)
    qr.make(fit=True)

    img = qr.make_image(fill_color="black", back_color="white")
    img = img.convert("RGB")
    img.save(save_path)

    return img


def read_qr_with_opencv(image_path):
    """Optional readability check using OpenCV."""
    if not CV2_AVAILABLE:
        return "not_checked", "OpenCV is not available"

    img = cv2.imread(str(image_path))
    detector = cv2.QRCodeDetector()
    decoded_text, points, _ = detector.detectAndDecode(img)

    if decoded_text:
        return "ok", decoded_text

    return "failed", "Could not decode"

In [ ]:
records = []

for raw_uid in USER_IDS:
    uid = sanitize_uid(raw_uid)
    url = make_user_url(BASE_URL, uid)

    filename = f"{uid}.png"
    filepath = OUTPUT_DIR / filename

    img = create_clean_qr(url, filepath)
    check_status, decoded_result = read_qr_with_opencv(filepath)

    records.append({
        "uid": uid,
        "url": url,
        "qr_file": filename,
        "decode_check": check_status,
        "decoded_text": decoded_result,
    })

In [ ]:
preview_count = min(len(records), 6)

for i in range(preview_count):
    row = records[i]
    img = Image.open(OUTPUT_DIR / row["qr_file"])

    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.title(row["uid"])
    plt.show()

In [ ]:
mapping_df = pd.DataFrame(records)
mapping_df.to_csv("qr_mapping_clean.csv", index=False, encoding="utf-8-sig")

print("Generated QR mapping:")
display(mapping_df)

In [ ]:
zip_filename = "qr_codes_clean.zip"

with zipfile.ZipFile(zip_filename, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    for png_file in OUTPUT_DIR.glob("*.png"):
        zipf.write(png_file, arcname=png_file.name)

    zipf.write("qr_mapping_clean.csv", arcname="qr_mapping_clean.csv")

print(f"Created: {zip_filename}")
files.download(zip_filename)